# Evaluation — Baseline vs Fine-tuned on Test Set
Evaluates both models on the held-out test split and saves all metrics, probabilities, and per-class numbers for Week 5 visualisation.

In [2]:
pip install transformers datasets scikit-learn -q

Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
import numpy as np
import pandas as pd
import os
import json
from datasets import load_from_disk
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSequenceClassification, default_data_collator
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
)

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
THRESHOLD  = 0.5
NUM_LABELS = 28
LABEL_NAMES = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
    'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
    'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
    'relief', 'remorse', 'sadness', 'surprise', 'neutral'
]
print('device:', DEVICE)

device: cpu


## 1. Load test split

In [10]:
tokenized = load_from_disk('processed/go_emotions_tokenized')
tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'token_type_ids'], output_all_columns=True)

class EmotionDataset(Dataset):
    def __init__(self, hf_split, num_labels=NUM_LABELS):
        self.data       = hf_split
        self.num_labels = num_labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        label_vec = torch.zeros(self.num_labels, dtype=torch.float)
        for lbl in row['labels']:
            label_vec[lbl] = 1.0
        return {
            'input_ids':      row['input_ids'],
            'attention_mask': row['attention_mask'],
            'token_type_ids': row['token_type_ids'],
            'labels':         label_vec,
        }

test_loader = DataLoader(
    EmotionDataset(tokenized['test']),
    batch_size=64, shuffle=False,
    collate_fn=default_data_collator,
)
print(f'Test examples: {len(tokenized["test"])}')

Test examples: 5427


## 2. Inference helper

In [4]:
def get_predictions(model, loader):
    """Returns (probs [N,28], labels [N,28]) on the given loader."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(
                input_ids      = batch['input_ids'].to(DEVICE),
                attention_mask = batch['attention_mask'].to(DEVICE),
                token_type_ids = batch['token_type_ids'].to(DEVICE),
            ).logits
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(batch['labels'].numpy())
    return np.vstack(all_probs), np.vstack(all_labels)

## 3. Load baseline model and run inference

In [5]:
baseline_model = AutoModelForSequenceClassification.from_pretrained(
    'bhadresh-savani/bert-base-go-emotion'
).to(DEVICE)

with open('checkpoints_initial/checkpoint-13570/trainer_state.json') as f:
    trainer_state = json.load(f)

INITIAL_CKPT = trainer_state['best_model_checkpoint']
print(f'Best checkpoint: {INITIAL_CKPT}')

initial_model = AutoModelForSequenceClassification.from_pretrained(INITIAL_CKPT).to(DEVICE)

with open('checkpoints_freeze3/checkpoint-13570/trainer_state.json') as f:
    trainer_state = json.load(f)

FREEZE3_CKPT = trainer_state['best_model_checkpoint']
print(f'Best checkpoint: {FREEZE3_CKPT}')

freeze3_model = AutoModelForSequenceClassification.from_pretrained(FREEZE3_CKPT).to(DEVICE)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bhadresh-savani/bert-base-go-emotion
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Best checkpoint: checkpoints_initial/checkpoint-12213


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Best checkpoint: checkpoints_freeze3/checkpoint-12213


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [12]:
baseline_model = AutoModelForSequenceClassification.from_pretrained(
    'bhadresh-savani/bert-base-go-emotion'
).to(DEVICE)

baseline_probs, test_labels = get_predictions(baseline_model, test_loader)
print(f'baseline_probs : {baseline_probs.shape}')
print(f'test_labels    : {test_labels.shape}')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bhadresh-savani/bert-base-go-emotion
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


baseline_probs : (5427, 28)
test_labels    : (5427, 28)


## 4. Load fine-tuned model and run inference
Reads `checkpoints/trainer_state.json` to find the best checkpoint automatically.
If you want to evaluate a specific run, set `FINETUNED_CKPT` manually.

In [16]:
with open('checkpoints_initial/checkpoint-13570/trainer_state.json') as f:
    trainer_state = json.load(f)

INITIAL_CKPT = trainer_state['best_model_checkpoint']
print(f'Best checkpoint: {INITIAL_CKPT}')

initial_model = AutoModelForSequenceClassification.from_pretrained(INITIAL_CKPT).to(DEVICE)
initial_probs, _ = get_predictions(initial_model, test_loader)
print(f'initial_probs: {initial_probs.shape}')

Best checkpoint: checkpoints_initial/checkpoint-12213


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

initial_probs: (5427, 28)


In [17]:
with open('checkpoints_freeze3/checkpoint-13570/trainer_state.json') as f:
    trainer_state = json.load(f)

FREEZE3_CKPT = trainer_state['best_model_checkpoint']
print(f'Best checkpoint: {FREEZE3_CKPT}')

freeze3_model = AutoModelForSequenceClassification.from_pretrained(FREEZE3_CKPT).to(DEVICE)
freeze3_probs, _ = get_predictions(freeze3_model, test_loader)
print(f'freeze3_probs: {freeze3_probs.shape}')

Best checkpoint: checkpoints_freeze3/checkpoint-12213


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

freeze3_probs: (5427, 28)


## 5. Compute all metrics

In [18]:
def compute_all_metrics(probs, labels, threshold=THRESHOLD):
    preds      = (probs >= threshold).astype(int)
    labels_int = labels.astype(int)

    overall = {
        'micro_f1':    float(f1_score(labels_int, preds, average='micro',    zero_division=0)),
        'macro_f1':    float(f1_score(labels_int, preds, average='macro',    zero_division=0)),
        'weighted_f1': float(f1_score(labels_int, preds, average='weighted', zero_division=0)),
        'exact_match': float((preds == labels_int).all(axis=1).mean()),
    }

    per_class = pd.DataFrame({
        'label':         LABEL_NAMES,
        'precision':     precision_score(labels_int, preds, average=None, zero_division=0),
        'recall':        recall_score(labels_int,    preds, average=None, zero_division=0),
        'f1':            f1_score(labels_int,        preds, average=None, zero_division=0),
        'support':       labels_int.sum(axis=0).astype(int),
        'auc_roc':  [
            float(roc_auc_score(labels_int[:, i], probs[:, i]))
            if labels_int[:, i].sum() > 0 else float('nan')
            for i in range(NUM_LABELS)
        ],
        'avg_precision': [
            float(average_precision_score(labels_int[:, i], probs[:, i]))
            if labels_int[:, i].sum() > 0 else float('nan')
            for i in range(NUM_LABELS)
        ],
    })

    return overall, per_class

In [19]:
baseline_overall,  baseline_pc  = compute_all_metrics(baseline_probs,  test_labels)
initial_overall, initial_pc = compute_all_metrics(initial_probs, test_labels)
freeze3_overall, freeze3_pc = compute_all_metrics(freeze3_probs, test_labels)

## 6. Comparison table

In [20]:
metrics = ['micro_f1', 'macro_f1', 'weighted_f1', 'exact_match']
summary = pd.DataFrame({
    'metric':    metrics,
    'baseline':  [baseline_overall[m]  for m in metrics],
    'initial':   [initial_overall[m]   for m in metrics],
    'freeze3':   [freeze3_overall[m]   for m in metrics],
})
summary['delta_initial'] = summary['initial'] - summary['baseline']
summary['delta_freeze3'] = summary['freeze3'] - summary['baseline']
summary['delta_pct_initial'] = (summary['delta_initial'] / summary['baseline'] * 100).round(2)
summary['delta_pct_freeze3'] = (summary['delta_freeze3'] / summary['baseline'] * 100).round(2)

pd.set_option('display.float_format', '{:.4f}'.format)
print('=== Test Set — Overall Metrics ===')
display(summary)

print('\n=== Per-Class Metrics (Fine-tuned, sorted by F1) ===')
display(initial_pc.sort_values('f1', ascending=False).reset_index(drop=True))
display(freeze3_pc.sort_values('f1', ascending=False).reset_index(drop=True))


=== Test Set — Overall Metrics ===


,metric,baseline,initial,freeze3,delta_initial,delta_freeze3,delta_pct_initial,delta_pct_freeze3
0,micro_f1,0.4795,0.5680,0.5763,0.0885,0.0968,18.4600,20.2000
1,macro_f1,0.2980,0.4797,0.5027,0.1818,0.2047,61.0100,68.7000
2,weighted_f1,0.4272,0.5657,0.5721,0.1385,0.1450,32.4300,33.9400
3,exact_match,0.3243,0.4540,0.4514,0.1297,0.1271,40.0000,39.2000



=== Per-Class Metrics (Fine-tuned, sorted by F1) ===


,label,precision,recall,f1,support,auc_roc,avg_precision
0,gratitude,0.9422,0.8807,0.9104,352,0.9901,0.9485
1,amusement,0.7661,0.8561,0.8086,264,0.9727,0.7812
2,love,0.7692,0.7983,0.7835,238,0.9727,0.7945
3,fear,0.6517,0.7436,0.6946,78,0.9676,0.6595
4,admiration,0.6774,0.7123,0.6944,504,0.9406,0.6951
5,remorse,0.5811,0.7679,0.6615,56,0.9910,0.5886
6,joy,0.6667,0.6211,0.6431,161,0.9417,0.6190
7,neutral,0.6662,0.5730,0.6161,1787,0.8095,0.6618
8,surprise,0.5846,0.5390,0.5609,141,0.9274,0.5197
9,optimism,0.5747,0.5376,0.5556,186,0.8899,0.4956


,label,precision,recall,f1,support,auc_roc,avg_precision
0,gratitude,0.9379,0.9006,0.9188,352,0.9884,0.9503
1,amusement,0.7348,0.8712,0.7972,264,0.9708,0.7892
2,love,0.7668,0.8151,0.7902,238,0.9793,0.7699
3,admiration,0.6514,0.7044,0.6768,504,0.9437,0.6823
4,neutral,0.6469,0.6603,0.6536,1787,0.8247,0.6831
5,fear,0.6044,0.7051,0.6509,78,0.9717,0.6831
6,remorse,0.5882,0.7143,0.6452,56,0.9807,0.6801
7,joy,0.6364,0.6087,0.6222,161,0.9330,0.6147
8,surprise,0.5436,0.5745,0.5586,141,0.9309,0.5361
9,optimism,0.6067,0.4892,0.5417,186,0.8775,0.4899


## 7. Save all results

In [21]:
os.makedirs('results', exist_ok=True)

# Overall metrics
with open('results/test_metrics_baseline.json',  'w') as f: json.dump(baseline_overall,  f, indent=2)
with open('results/test_metrics_initial.json', 'w') as f: json.dump(initial_overall, f, indent=2)
with open('results/test_metrics_freeze3.json', 'w') as f: json.dump(freeze3_overall, f, indent=2)
# Per-class metrics
baseline_pc.to_csv( 'results/test_per_class_baseline.csv',  index=False)
initial_pc.to_csv('results/test_per_class_initial.csv', index=False)
freeze3_pc.to_csv('results/test_per_class_freeze3.csv', index=False)

# Raw probabilities and labels (needed for ROC / PR curves)
np.save('results/test_probs_baseline.npy',  baseline_probs)
np.save('results/test_probs_initial.npy', initial_probs)
np.save('results/test_probs_freeze3.npy', freeze3_probs)
np.save('results/test_labels.npy',          test_labels)

print('Saved to results/:')
for fname in sorted(os.listdir('results')):
    print(f'  {fname}')

Saved to results/:
  best_freeze3.json
  best_initial.json
  run_freeze3.csv
  run_initial.csv
  summary
  test_labels.npy
  test_metrics_baseline.json
  test_metrics_freeze3.json
  test_metrics_initial.json
  test_per_class_baseline.csv
  test_per_class_freeze3.csv
  test_per_class_initial.csv
  test_probs_baseline.npy
  test_probs_freeze3.npy
  test_probs_initial.npy


## 8. Sentence-Sequence Prediction Demo
Runs three emotionally shifting sentences through all three models to illustrate how predictions evolve across context — used in the Discussion section of the poster.

In [7]:
from transformers import AutoTokenizer

TOKENIZER = AutoTokenizer.from_pretrained('bert-base-uncased')
TOP_K = 3

SENTENCES = [
    "I was excited to start this.",
    "But things quickly went wrong.",
    "Now I just feel numb.",
]

EVAL_MODELS = [
    ('Baseline',  baseline_model),
    ('Initial',   initial_model),
    ('Freeze-3',  freeze3_model),
]

def predict_topk(model, sentence, k=TOP_K):
    inputs = TOKENIZER(sentence, return_tensors='pt', truncation=True, max_length=128).to(DEVICE)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.sigmoid(logits).squeeze().cpu().numpy()
    top_idxs = probs.argsort()[::-1][:k]
    return [(LABEL_NAMES[i], round(float(probs[i]), 3)) for i in top_idxs]

rows = []
for sentence in SENTENCES:
    for model_name, model in EVAL_MODELS:
        preds = predict_topk(model, sentence)
        rows.append({
            'Sentence': sentence,
            'Model': model_name,
            **{f'Top-{i+1}': f"{label} ({score:.3f})" for i, (label, score) in enumerate(preds)},
        })

demo_df = pd.DataFrame(rows)
display(demo_df.to_string(index=False))


'                      Sentence    Model                  Top-1               Top-2                  Top-3\n  I was excited to start this. Baseline     excitement (0.702)         joy (0.162)       surprise (0.090)\n  I was excited to start this.  Initial     excitement (0.964)         joy (0.021)       approval (0.018)\n  I was excited to start this. Freeze-3     excitement (0.996)         joy (0.026)      amusement (0.012)\nBut things quickly went wrong. Baseline        neutral (0.345) disapproval (0.235)       approval (0.152)\nBut things quickly went wrong.  Initial disappointment (0.779) realization (0.534)        sadness (0.024)\nBut things quickly went wrong. Freeze-3 disappointment (0.937)     sadness (0.459)    realization (0.105)\n         Now I just feel numb. Baseline           fear (0.332) nervousness (0.182)        sadness (0.162)\n         Now I just feel numb.  Initial        neutral (0.870)     sadness (0.219) disappointment (0.075)\n         Now I just feel numb. Freez